# Clustering with sentence embeddings

This notebook treats clustering as an exploratory research tool. You will first fit
a clustering solution for UK parliamentary sentences. We then switch to a shared
prepared solution, inspect one cluster closely, propose a substantive
interpretation, and only afterward compare that interpretation with the Comparative
Agendas Project (CAP) topic codes.

By the end, you should be able to

- interpret HDBSCAN cluster and noise labels;
- support a provisional cluster label with original texts and distinctive terms;
- identify atypical or ambiguous cluster members;
- use an external coding scheme as evidence without treating it as unquestionable
  ground truth;
- distinguish the representation used for clustering from a two-dimensional map.

<br><a target="_blank" href="https://colab.research.google.com/github/haukelicht/advanced_text_analysis/blob/main/notebooks/embedding/sentence_embedding_clustering.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Note:** If running on Google Colab, make sure to use a GPU runtime (go to Runtime > Change runtime type, select "T4 GPU", and click save). See [this guide](https://github.com/haukelicht/advanced_text_analysis/blob/main/setup/setup_colab_runtime.md).

## Setup

In [ ]:
COLAB = True
try:
    import google.colab
except ImportError:
    COLAB = False

if COLAB:
    !git clone --branch main --single-branch --depth 1 --filter=blob:none https://github.com/haukelicht/advanced_text_analysis.git
    !pip install -q sentence-transformers~=6.1.0 umap-learn~=0.5.12 seaborn~=0.13.2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score
from umap import UMAP

In [ ]:
SEED = 42

base_path = Path("/content/advanced_text_analysis/" if COLAB else "../../")

data_path = base_path / "data/labeled/sylvester_parlee_2022"
data_file = "sylvester_parlee_2022-uk_cap_sentences.csv"

## Load the sentence corpus

The prepared file contains one row per unique sentence and a stable `text_id`. The
CAP code remains in the data for the later reveal; it is not supplied to the
embedding or clustering model.

In [ ]:
fp = data_path / data_file
if not fp.exists():
    url = (
        "https://cta-text-datasets.s3.eu-central-1.amazonaws.com/labeled/"
        f"{data_path.parent.name}/{data_file}"
    )
    df = pd.read_csv(url)
    df.to_csv(fp, index=False)

In [ ]:
df = pd.read_csv(fp)
df = df.dropna(subset=["text_id", "text", "cap_topic"]).reset_index(drop=True)

df[["text_id", "text"]].head()

## Fitting a clustering solution

You will build the complete clustering pipeline in four steps:

1. encode each sentence with `all-MiniLM-L6-v2`;
2. reduce the 384-dimensional embeddings to 24 dimensions with UMAP;
3. fit HDBSCAN in that 24-dimensional space;
4. create a separate two-dimensional UMAP only for display.

::: {.callout-tip title="Using cached results (instead)"}

From section [Fitting a clustering solution](#fitting-a-clustering-solution) onwards, we will use cached clustering results. 
You may thus skip the next steps showing how to fit the clustering solution.
But first try it yourself 😉

:::


HDBSCAN assigns documents in sufficiently dense regions to clusters. Label `-1`
means that a document was not assigned to any cluster and is treated as noise. The
cluster numbers themselves have no ordering or substantive meaning.

::: {.callout-warning title="The map is not the clustering space"}

Fit the clusters in the 24-dimensional reduced space. Create the two-dimensional
coordinates separately and use them only for visualization. Apparent distances,
gaps, and overlaps on the map can be projection artifacts.

:::

In [ ]:
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

CLUSTERING_PARAMETERS = {
    "model_id": MODEL_ID,
    "umap_components_for_clustering": 24,
    "umap_n_neighbors": 15,
    "umap_min_dist": 0.0,
    "hdbscan_min_cluster_size": 40,
    "hdbscan_min_samples": 10,
    "seed": SEED,
}

### Step 1: encode every sentence

The encoder converts every sentence into a 384-dimensional vector. Normalizing the
vectors is useful for cosine-based comparisons and keeps this representation
consistent with the retrieval exercise.

In [ ]:
# TODO: Load the specified sentence-embedding model.
embedding_model = SentenceTransformer(MODEL_ID)

In [ ]:
# TODO: Encode every sentence. This may take a few minutes on a CPU.
embeddings = embedding_model.encode(
    df["text"].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)

assert embeddings.shape == (len(df), 384)
embeddings.shape

### Step 2: create the clustering representation

Clustering thousands of dense vectors directly can be difficult. UMAP creates a
lower-dimensional representation intended for the clustering algorithm. Twenty-four
dimensions retain considerably more structure than a two-dimensional plot.

In [ ]:
# TODO: Configure UMAP to produce 24 dimensions and fit it to the embeddings.
clustering_reducer = UMAP(
    n_components=CLUSTERING_PARAMETERS["umap_components_for_clustering"],
    n_neighbors=CLUSTERING_PARAMETERS["umap_n_neighbors"],
    min_dist=CLUSTERING_PARAMETERS["umap_min_dist"],
    metric="cosine",
    random_state=SEED,
    n_jobs=1,
)
clustering_coordinates = clustering_reducer.fit_transform(embeddings)

assert clustering_coordinates.shape == (len(df), 24)
clustering_coordinates.shape

### Step 3: fit HDBSCAN

`min_cluster_size` sets the smallest group that HDBSCAN should treat as a cluster.
`min_samples` influences how conservative the density criterion is. We use fixed
values so that everyone begins with the same modeling choices.

In [ ]:
# TODO: Fit HDBSCAN in the 24-dimensional clustering space.
clusterer = HDBSCAN(
    min_cluster_size=CLUSTERING_PARAMETERS["hdbscan_min_cluster_size"],
    min_samples=CLUSTERING_PARAMETERS["hdbscan_min_samples"],
    metric="euclidean",
    cluster_selection_method="eom",
    n_jobs=1,
    copy=True,
)
cluster_labels = clusterer.fit_predict(clustering_coordinates)

pd.Series(cluster_labels).value_counts().sort_index()

Before continuing, check how many clusters were found and how many sentences were
treated as noise.

In [ ]:
n_trial_clusters = len(set(cluster_labels)) - int(-1 in cluster_labels)
trial_noise_share = np.mean(cluster_labels == -1)

print(f"Clusters: {n_trial_clusters}")
print(f"Share labeled as noise: {trial_noise_share:.1%}")

### Step 4: create a separate two-dimensional display

This UMAP is fitted directly to the sentence embeddings and is used only to draw a
map. It does not determine the HDBSCAN labels.

In [ ]:
# TODO: Fit a separate two-dimensional UMAP for display.
display_reducer = UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=SEED,
    n_jobs=1,
)
display_coordinates = display_reducer.fit_transform(embeddings)

assert display_coordinates.shape == (len(df), 2)
display_coordinates.shape

Combine the labels, membership strengths, and display coordinates with the original
sentences. This is your own fitted solution.

In [ ]:
trial_results = df[["text_id", "text", "party", "cap_topic"]].copy()
trial_results["cluster"] = cluster_labels
trial_results["membership_strength"] = clusterer.probabilities_
trial_results["display_x"] = display_coordinates[:, 0]
trial_results["display_y"] = display_coordinates[:, 1]

trial_results.head()

## Inspect the clustering solution {#inspect-clustering-solution}

::: {.callout-warning title="Load the cached results"}

From this point onward, use the cached results so that everyone audits the same
clustering solution. Your own fitted solution remains available as `trial_results`.

In [ ]:
results_path = base_path / "results/sentence_embedding_clustering"
results = pd.read_csv(results_path / "clustering_results.csv")

required_result_cols = {
    "text_id", "text", "cap_topic", "cluster", "membership_strength",
    "display_x", "display_y",
}
if not required_result_cols.issubset(results.columns):
    raise ValueError("The cached result file is missing required columns.")
if results["text_id"].tolist() != df["text_id"].tolist():
    raise ValueError("Cached results are not aligned with the current corpus.")

:::

In [ ]:
cluster_summary = (
    results["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="n_sentences")
)
cluster_summary["kind"] = np.where(
    cluster_summary["cluster"].eq(-1), "noise", "cluster"
)
cluster_summary

In [ ]:
n_clusters = results.loc[results["cluster"].ge(0), "cluster"].nunique()
noise_share = results["cluster"].eq(-1).mean()
print(f"Clusters: {n_clusters}")
print(f"Share labeled as noise: {noise_share:.1%}")

## Audit one cluster

Every group audits the same prepared cluster. Cluster 23 was selected because its
members support a plausible housing-related interpretation while also containing
less typical and ambiguous cases.

In [ ]:
AUDIT_CLUSTER = 23
if AUDIT_CLUSTER not in set(results["cluster"]):
    raise ValueError(
        "The prepared audit cluster is absent. "
        f"Load the cached results from {results_path / 'clustering_results.csv'}."
    )

audit_members = results.query("cluster == @AUDIT_CLUSTER").copy()

len(audit_members)

### View the cluster on the corpus map

In [ ]:
is_audit = results["cluster"].eq(AUDIT_CLUSTER)
is_noise = results["cluster"].eq(-1)

plt.figure(figsize=(9, 7))
plt.scatter(
    results.loc[~is_audit & ~is_noise, "display_x"],
    results.loc[~is_audit & ~is_noise, "display_y"],
    color="#BBBBBB", s=8, alpha=0.35, label="other clusters",
)
plt.scatter(
    results.loc[is_noise, "display_x"],
    results.loc[is_noise, "display_y"],
    color="#666666", marker="x", s=8, alpha=0.25, label="noise",
)
plt.scatter(
    results.loc[is_audit, "display_x"],
    results.loc[is_audit, "display_y"],
    color="#D55E00", s=20, alpha=0.85, label=f"cluster {AUDIT_CLUSTER}",
)
plt.xlabel("UMAP display dimension 1")
plt.ylabel("UMAP display dimension 2")
plt.title("Prepared clustering with the shared audit cluster highlighted")
plt.legend()
plt.show()

### Read typical and less typical members

HDBSCAN's membership strength describes how strongly a sentence belongs to its
assigned cluster under this fitted solution. It is useful for selecting cases to
inspect, but it is not a probability that the proposed substantive label is true.

In [ ]:
typical_members = audit_members.nlargest(3, "membership_strength")
less_typical_members = audit_members.nsmallest(2, "membership_strength")

audit_examples = pd.concat(
    [
        typical_members.assign(example_type="higher membership strength"),
        less_typical_members.assign(example_type="lower membership strength"),
    ],
    ignore_index=True,
).drop_duplicates("text_id")

audit_examples = audit_examples[
    ["example_type", "text_id", "text", "membership_strength", "cap_topic"]
].sort_values("membership_strength", ascending=False)

from IPython.display import display, HTML
display(HTML(audit_examples.to_html()))

### Inspect a few unassigned sentences

Noise is not synonymous with bad data. A sentence can be unassigned because it is
unusual, underspecified, situated between dense regions, or part of a small theme.

In [ ]:
noise_examples = (
    results.query("cluster == -1")
    .sample(3, random_state=SEED)
    [["text_id", "text"]]
)
noise_examples

### Describe the cluster with distinctive terms

Representative texts should drive interpretation. A short term list is supporting
evidence that helps compare the vocabulary of this cluster with the full corpus.
The score below rewards terms that are frequent within the cluster and downweights
terms that occur in many documents throughout the corpus.

In [ ]:
vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'-]+\b",
)
document_term_matrix = vectorizer.fit_transform(results["text"])
terms = vectorizer.get_feature_names_out()
document_frequency = np.asarray((document_term_matrix > 0).sum(axis=0)).ravel()
inverse_document_frequency = np.log(
    (len(results) + 1) / (document_frequency + 1)
)

audit_counts = np.asarray(
    document_term_matrix[is_audit.to_numpy()].sum(axis=0)
).ravel()
audit_term_scores = (
    audit_counts / audit_counts.sum()
) * inverse_document_frequency

top_term_indices = np.argsort(audit_term_scores)[::-1][:15]
pd.DataFrame({
    "term": terms[top_term_indices],
    "score": audit_term_scores[top_term_indices],
})

::: {.callout-warning title="Terms describe; they do not validate"}

Top terms are computed after the documents have been clustered. They can support a
provisional interpretation but cannot establish that a cluster is coherent, valid,
or equivalent to a theoretical concept.

:::

## Exercise: record a blind cluster audit

Complete this audit before revealing the CAP topic codes.

In [ ]:
proposed_label = "TODO"

evidence_for_label = """
TODO: Which sentences and terms support this label?
""".strip()

ambiguous_or_conflicting_evidence = """
TODO: Which members fit poorly or suggest another interpretation?
""".strip()

assessment = """
TODO: Is this cluster coherent and useful for a research application? Why or why not?
""".strip()

## Reveal and compare the CAP topic codes

CAP provides an external human coding scheme. Agreement can support an
interpretation, while disagreement can indicate that the cluster subdivides a CAP
topic, combines several topics, captures another textual feature, or contains
difficult cases.

In [ ]:
codebook_file = data_path / "cap_topic_codes.tsv"
codebook = pd.read_csv(codebook_file, sep="\t")

cap_topic_names = dict(zip(codebook["code"], codebook["topic"]))
cap_topic_names[99] = "No policy content / other"

In [ ]:
revealed_examples = audit_examples[
    ["example_type", "text_id", "text", "membership_strength", "cap_topic"]
].copy()
revealed_examples["cap_topic_name"] = revealed_examples["cap_topic"].map(cap_topic_names)
revealed_examples

In [ ]:
audit_cap_distribution = (
    audit_members["cap_topic"]
    .value_counts(normalize=True)
    .rename("share")
    .rename_axis("cap_topic")
    .reset_index()
)
audit_cap_distribution["cap_topic_name"] = audit_cap_distribution["cap_topic"].map(
    cap_topic_names
)
audit_cap_distribution

In [ ]:
cluster_cap_table = pd.crosstab(
    results.loc[results["cluster"].ge(0), "cluster"],
    results.loc[results["cluster"].ge(0), "cap_topic"],
    normalize="index",
)

plt.figure(figsize=(12, 8))
sns.heatmap(cluster_cap_table, cmap="Blues", vmin=0, vmax=1)
plt.xlabel("CAP topic code")
plt.ylabel("Embedding cluster")
plt.title("CAP-topic composition within each prepared cluster")
plt.show()

After the reveal, revisit the blind audit:

- Where does the proposed label agree with the CAP evidence?
- Does the cluster subdivide or combine CAP topics?
- Do the less typical members reveal a limitation of the label or of the clustering?
- What additional validation would be needed before using the cluster as a
  measurement in research?

## Optional appendix: fitting and robustness choices

The fixed parameters above were chosen for a manageable teaching example. They do
not reveal a true number of topics. HDBSCAN results can change with the embedding
model, dimensionality reduction, `min_cluster_size`, `min_samples`, distance
metric, and random seed.

The following optional code compares a few HDBSCAN settings. It is intended for
later exploration or a robustness analysis and is not part of the core exercise.

In [ ]:
#| eval: false
robustness_rows = []
for min_cluster_size in [30, 40, 50]:
    for min_samples in [5, 10, 20]:
        candidate = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="euclidean",
            cluster_selection_method="eom",
            n_jobs=1,
            copy=True,
        ).fit(clustering_coordinates)
        robustness_rows.append({
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "n_clusters": len(set(candidate.labels_)) - int(-1 in candidate.labels_),
            "noise_share": np.mean(candidate.labels_ == -1),
        })

pd.DataFrame(robustness_rows)

K-means represents a different modeling choice: it requires a fixed number of
clusters and assigns every document. This compact comparison is also optional.

In [ ]:
#| eval: false
kmeans_labels = KMeans(n_clusters=20, random_state=SEED, n_init="auto").fit_predict(
    clustering_coordinates
)
pd.Series(kmeans_labels).value_counts().sort_index()

If CAP labels are available, permutation-invariant measures such as adjusted Rand
index and adjusted mutual information can describe alignment with that external
scheme. They do not establish substantive validity, because CAP and the embedding
clusters may capture different distinctions.

In [ ]:
#| eval: false
assigned = results["cluster"].ge(0)
pd.Series({
    "adjusted Rand index": adjusted_rand_score(
        results.loc[assigned, "cap_topic"], results.loc[assigned, "cluster"]
    ),
    "adjusted mutual information": adjusted_mutual_info_score(
        results.loc[assigned, "cap_topic"], results.loc[assigned, "cluster"]
    ),
})